# fMRI Analysis Pipeline: Mixed-Gambles Task (ds000005)

## Tom et al. (2007) — *The Neural Basis of Loss Aversion in Decision-Making Under Risk*

This notebook demonstrates a **complete fMRI analysis pipeline** using the OpenNeuro ds000005 dataset:

1. **Data download** from OpenNeuro (BIDS format)
2. **Preprocessing check** (using fMRIPrep derivatives)
3. **First-level GLM** — individual subject modeling with parametric modulators
4. **Second-level GLM** — group-level statistical analysis

### About the Study
- **16 subjects** viewed 256 mixed gambles (50/50 chance of gaining \$10–\$40 or losing \$5–\$20)
- Each trial: participant decides to **accept** or **reject** the gamble
- **3 functional runs** per subject
- Key finding: loss-related neural signals are **steeper** than gain-related signals (neural loss aversion)

### Do I need to run fMRIPrep?
**No!** Preprocessed derivatives are available from OpenNeuroDerivatives.  
fMRIPrep handles: motion correction, slice-timing correction, spatial normalization to MNI space, confound estimation.  
We download these results directly.

---

## 0. Setup & Installation

In [ ]:
# Install required packages (uncomment if needed)
# !pip install nilearn nibabel matplotlib pandas numpy scipy openneuro-py

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

from nilearn import plotting, image, datasets
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.glm.second_level import SecondLevelModel
from nilearn.reporting import make_glm_report

print('All imports successful!')

In [ ]:
# Data directory — change this to your preferred location
DATA_DIR = os.path.expanduser('~/nilearn_data/ds000005')
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Data will be stored in: {DATA_DIR}')

---
## 1. Download Data from OpenNeuro

We download the **raw BIDS data** (for events files and anatomical images)  
and the **fMRIPrep derivatives** (preprocessed functional data in MNI space).

### Option A: Using openneuro-py (recommended)

In [ ]:
# Option A: Download via openneuro-py CLI
# This downloads a subset of the dataset (3 subjects for quick demo)
# For the full dataset, remove the --include flags

# !openneuro-py download --dataset ds000005 --target-dir {DATA_DIR}/raw \
#     --include sub-01 --include sub-02 --include sub-03 \
#     --include dataset_description.json --include task-mixedgamblestask_bold.json

### Option B: Using DataLad (better for full dataset)

In [ ]:
# Option B: Using datalad
# !pip install datalad
# !datalad install https://github.com/OpenNeuroDatasets/ds000005.git {DATA_DIR}/raw
# !datalad install https://github.com/OpenNeuroDerivatives/ds000005-fmriprep.git {DATA_DIR}/derivatives/fmriprep

### Option C: Using Nilearn's built-in fetcher

For this tutorial, we'll use **Nilearn's dataset fetching** which handles download automatically.  
We fetch a few subjects to keep things manageable.

In [ ]:
# Nilearn can fetch OpenNeuro datasets via URL lists
# For ds000005, we construct the download manually
# This cell sets up paths — adjust if you downloaded data via Option A or B

RAW_DIR = os.path.join(DATA_DIR, 'raw')
DERIV_DIR = os.path.join(DATA_DIR, 'derivatives', 'fmriprep')

# List of subjects to analyze
SUBJECTS = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05',
            'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10',
            'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16']

# For quick testing, use only a subset
SUBJECTS_DEMO = SUBJECTS[:5]  # first 5 subjects

# Number of runs per subject
N_RUNS = 3

# Task name as in BIDS
TASK = 'mixedgamblestask'

print(f'Raw data dir:    {RAW_DIR}')
print(f'Derivatives dir: {DERIV_DIR}')
print(f'Subjects: {len(SUBJECTS)}, Runs per subject: {N_RUNS}')

---
## 2. Explore the BIDS Structure

Let's understand how the data is organized.

In [ ]:
# Expected BIDS structure:
print("""
ds000005/
├── raw/                              # Raw BIDS dataset
│   ├── dataset_description.json
│   ├── participants.tsv
│   ├── sub-01/
│   │   ├── anat/
│   │   │   └── sub-01_T1w.nii.gz
│   │   └── func/
│   │       ├── sub-01_task-mixedgamblestask_run-01_bold.nii.gz
│   │       ├── sub-01_task-mixedgamblestask_run-01_events.tsv   <-- KEY FILE
│   │       ├── sub-01_task-mixedgamblestask_run-02_bold.nii.gz
│   │       ├── sub-01_task-mixedgamblestask_run-02_events.tsv
│   │       ├── sub-01_task-mixedgamblestask_run-03_bold.nii.gz
│   │       └── sub-01_task-mixedgamblestask_run-03_events.tsv
│   ├── sub-02/
│   └── ...
│
└── derivatives/fmriprep/             # fMRIPrep preprocessed data
    ├── sub-01/
    │   └── func/
    │       ├── sub-01_task-mixedgamblestask_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
    │       ├── sub-01_task-mixedgamblestask_run-01_desc-confounds_timeseries.tsv
    │       └── ...
    └── ...
""")

---
## 3. Examine the Events Files

The events files are **crucial** — they describe what happened on each trial.  
For the mixed-gambles task, each trial has a **gain amount** and **loss amount** as parametric modulators.

In [ ]:
# Load an example events file
events_file = os.path.join(RAW_DIR, 'sub-01', 'func',
                           'sub-01_task-mixedgamblestask_run-01_events.tsv')

if os.path.exists(events_file):
    events = pd.read_csv(events_file, sep='\t')
    print(f'Events file shape: {events.shape}')
    print(f'\nColumns: {list(events.columns)}')
    print(f'\nFirst 10 trials:')
    display(events.head(10))
else:
    print(f'Events file not found at: {events_file}')
    print('Please download the data first (see Section 1).')
    print('\nShowing expected structure:')
    # Create example events for demonstration
    events = pd.DataFrame({
        'onset': [0.0, 4.0, 8.0, 12.0, 16.0, 20.0, 24.0, 28.0, 32.0, 36.0],
        'duration': [3.0]*10,
        'gain': [14, 34, 38, 10, 16, 12, 20, 24, 22, 30],
        'loss': [6, 14, 19, 15, 17, 6, 16, 18, 10, 12],
        'RT': [1.556, 1.658, 1.335, 1.898, 1.442, 1.221, 1.767, 1.398, 1.556, 1.312],
        'response': [1, 1, 1, 0, 0, 1, 1, 0, 1, 1]
    })
    display(events)

In [ ]:
# Visualize the distribution of gains and losses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gain distribution
axes[0].hist(events['gain'], bins=15, color='green', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Potential Gain ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Gains')

# Loss distribution
axes[1].hist(events['loss'], bins=15, color='red', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Potential Loss ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Losses')

# Accept/Reject
if 'response' in events.columns:
    accept_rate = events['response'].mean() * 100
    axes[2].bar(['Reject', 'Accept'],
               [events['response'].value_counts().get(0, 0),
                events['response'].value_counts().get(1, 0)],
               color=['red', 'green'], alpha=0.7, edgecolor='black')
    axes[2].set_ylabel('Count')
    axes[2].set_title(f'Responses (Accept rate: {accept_rate:.1f}%)')

plt.tight_layout()
plt.show()

---
## 4. Preprocessing Overview

### What fMRIPrep already did for us:

| Step | Description |
|------|------------|
| **Skull stripping** | Removed non-brain tissue from anatomical image |
| **Motion correction** | Realigned all volumes to a reference volume |
| **Slice-timing correction** | Corrected for differences in slice acquisition time |
| **Coregistration** | Aligned functional images to anatomical image |
| **Spatial normalization** | Warped to MNI152 standard space |
| **Confound estimation** | Computed motion parameters, CSF/WM signals, etc. |

### What we still need to do:
- **Spatial smoothing** (applied during first-level modeling in Nilearn)
- **High-pass filtering** (handled by the design matrix drift terms)

Let's inspect the preprocessed data.

In [ ]:
# Check for preprocessed functional data
example_func = os.path.join(
    DERIV_DIR, 'sub-01', 'func',
    'sub-01_task-mixedgamblestask_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz'
)

if os.path.exists(example_func):
    img = nib.load(example_func)
    print(f'Preprocessed fMRI image:')
    print(f'  Shape: {img.shape}')  # (x, y, z, time)
    print(f'  Voxel size: {img.header.get_zooms()[:3]} mm')
    print(f'  TR: {img.header.get_zooms()[3]} s')
    print(f'  Number of volumes: {img.shape[3]}')
    
    # Plot the mean functional image
    mean_img = image.mean_img(img)
    plotting.plot_epi(mean_img, title='Mean Functional Image (MNI space)',
                      display_mode='ortho', cut_coords=(0, 0, 0))
    plt.show()
else:
    print(f'Preprocessed data not found at: {example_func}')
    print('Please download fMRIPrep derivatives (see Section 1).')
    print('\nContinuing with demonstration using synthetic example...')

In [ ]:
# Examine confounds from fMRIPrep
confounds_file = os.path.join(
    DERIV_DIR, 'sub-01', 'func',
    'sub-01_task-mixedgamblestask_run-01_desc-confounds_timeseries.tsv'
)

if os.path.exists(confounds_file):
    confounds_all = pd.read_csv(confounds_file, sep='\t')
    print(f'Available confound regressors ({len(confounds_all.columns)}):')
    print(list(confounds_all.columns)[:20], '...')
    
    # We typically use a subset of confounds
    print('\n--- Recommended confound selection ---')
    print('Motion (6 params): trans_x, trans_y, trans_z, rot_x, rot_y, rot_z')
    print('Tissue signals: csf, white_matter')
    print('High-motion volumes: motion_outlier columns (scrubbing)')
else:
    print('Confounds file not found. Demonstrating expected structure.')
    print('\nfMRIPrep confounds typically include:')
    print('  - 6 motion parameters (translations + rotations)')
    print('  - CSF and white matter mean signals')
    print('  - Framewise displacement')
    print('  - CompCor components')
    print('  - Motion outlier flags')

### Confound Selection Strategy

Nilearn provides `load_confounds()` to easily select confound regressors from fMRIPrep output:

In [ ]:
from nilearn.interfaces.fmriprep import load_confounds

# Example: load confounds with a standard strategy
# This selects motion parameters + WM/CSF signals + high-pass filtering cosines
if os.path.exists(example_func):
    confounds, sample_mask = load_confounds(
        example_func,
        strategy=['motion', 'wm_csf'],  # motion 6 params + WM/CSF signals
        motion='basic',                   # 6 basic motion parameters
        wm_csf='basic'                    # mean WM and CSF signals
    )
    print(f'Selected confounds: {list(confounds.columns)}')
    print(f'Shape: {confounds.shape}')
else:
    print('Skipping — preprocessed data not available.')
    print('When data is available, load_confounds() returns a DataFrame of selected regressors.')

---
## 5. First-Level GLM (Single Subject)

### The General Linear Model (GLM)

$$Y = X\beta + \epsilon$$

- **Y** = observed BOLD signal at each voxel (time series)
- **X** = design matrix (task regressors convolved with HRF + confounds)
- **β** = parameter estimates (how much each regressor explains the signal)
- **ε** = residual noise

### Our Design Matrix

For each run, we model:
1. **Gamble onset** — a regressor for when any gamble appears
2. **Parametric gain** — modulated by the gain amount (mean-centered)
3. **Parametric loss** — modulated by the loss amount (mean-centered)

Plus confound regressors (motion, drift).

In [ ]:
def prepare_events_for_glm(events_df):
    """
    Prepare events DataFrame for Nilearn's FirstLevelModel.
    
    Creates three trial types:
    - 'gamble': all gamble onsets (unmodulated)
    - 'gain': parametric modulator for gain amount
    - 'loss': parametric modulator for loss amount
    
    The gain and loss regressors use the parametric values as modulation.
    """
    # Mean-center the parametric values
    gain_centered = events_df['gain'] - events_df['gain'].mean()
    loss_centered = events_df['loss'] - events_df['loss'].mean()
    
    # Create events for the GLM
    glm_events = pd.DataFrame({
        'onset': events_df['onset'],
        'duration': events_df['duration'],
        'trial_type': 'gamble'
    })
    
    # Add parametric modulators as separate trial types with modulation values
    gain_events = pd.DataFrame({
        'onset': events_df['onset'],
        'duration': events_df['duration'],
        'trial_type': 'gain',
        'modulation': gain_centered
    })
    
    loss_events = pd.DataFrame({
        'onset': events_df['onset'],
        'duration': events_df['duration'],
        'trial_type': 'loss',
        'modulation': loss_centered
    })
    
    all_events = pd.concat([glm_events, gain_events, loss_events], 
                           ignore_index=True)
    return all_events


# Demo with our example events
glm_events = prepare_events_for_glm(events)
print('GLM events structure:')
print(f'Trial types: {glm_events["trial_type"].unique()}')
print(f'Total rows: {len(glm_events)}')
display(glm_events.head(15))

In [ ]:
# Visualize the design matrix for one run
from nilearn.glm.first_level import make_first_level_design_matrix

# Assume TR = 2s and ~140 volumes per run
TR = 2.0
n_volumes = 140
frame_times = np.arange(n_volumes) * TR

# Build design matrix
design_matrix = make_first_level_design_matrix(
    frame_times=frame_times,
    events=glm_events,
    hrf_model='spm',           # canonical SPM HRF
    drift_model='cosine',      # cosine drift (high-pass filter)
    high_pass=0.01             # 100s high-pass cutoff (~0.01 Hz)
)

print(f'Design matrix shape: {design_matrix.shape}')
print(f'Columns: {list(design_matrix.columns)}')

# Plot the design matrix
from nilearn.plotting import plot_design_matrix
fig, ax = plt.subplots(figsize=(8, 10))
plot_design_matrix(design_matrix, ax=ax)
ax.set_title('First-Level Design Matrix (1 run)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot the expected BOLD response for each condition
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for i, col in enumerate(['gamble', 'gain', 'loss']):
    if col in design_matrix.columns:
        axes[i].plot(frame_times, design_matrix[col], 
                     color=['blue', 'green', 'red'][i], linewidth=1.5)
        axes[i].set_ylabel('Predicted\nBOLD', fontsize=11)
        axes[i].set_title(f'{col.upper()} regressor (convolved with HRF)', fontsize=12)
        axes[i].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

axes[2].set_xlabel('Time (seconds)', fontsize=11)
plt.tight_layout()
plt.show()

### 5.1 Fit First-Level Model for One Subject

In [ ]:
def get_subject_data(subject_id, raw_dir, deriv_dir, task, n_runs):
    """
    Collect all files needed for first-level analysis of one subject.
    
    Returns:
        func_imgs: list of preprocessed fMRI images (one per run)
        events_list: list of events DataFrames (one per run)
        confounds_list: list of confound DataFrames (one per run)
    """
    func_imgs = []
    events_list = []
    confounds_list = []
    
    for run in range(1, n_runs + 1):
        run_str = f'run-{run:02d}'
        
        # Preprocessed functional image (MNI space)
        func_file = os.path.join(
            deriv_dir, subject_id, 'func',
            f'{subject_id}_task-{task}_{run_str}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz'
        )
        
        # Events file (from raw BIDS)
        events_file = os.path.join(
            raw_dir, subject_id, 'func',
            f'{subject_id}_task-{task}_{run_str}_events.tsv'
        )
        
        # Confounds from fMRIPrep
        confounds_file = os.path.join(
            deriv_dir, subject_id, 'func',
            f'{subject_id}_task-{task}_{run_str}_desc-confounds_timeseries.tsv'
        )
        
        if os.path.exists(func_file) and os.path.exists(events_file):
            func_imgs.append(func_file)
            
            # Load and prepare events
            ev = pd.read_csv(events_file, sep='\t')
            events_list.append(prepare_events_for_glm(ev))
            
            # Load confounds (select key regressors)
            if os.path.exists(confounds_file):
                conf = pd.read_csv(confounds_file, sep='\t')
                # Select motion parameters + tissue signals
                conf_cols = [c for c in conf.columns if c in [
                    'trans_x', 'trans_y', 'trans_z',
                    'rot_x', 'rot_y', 'rot_z',
                    'csf', 'white_matter'
                ]]
                confounds_list.append(conf[conf_cols].fillna(0))
            else:
                confounds_list.append(None)
        else:
            print(f'  Warning: Missing files for {subject_id} {run_str}')
    
    return func_imgs, events_list, confounds_list


print('Function defined. Ready to load subject data.')

In [ ]:
def fit_first_level(subject_id, raw_dir, deriv_dir, task, n_runs,
                    smoothing_fwhm=6.0, tr=2.0):
    """
    Fit a first-level GLM for one subject.
    
    Parameters:
        smoothing_fwhm: spatial smoothing in mm (6mm is standard)
        tr: repetition time in seconds
    
    Returns:
        first_level_model: fitted FirstLevelModel
    """
    func_imgs, events_list, confounds_list = get_subject_data(
        subject_id, raw_dir, deriv_dir, task, n_runs
    )
    
    if not func_imgs:
        print(f'  No data found for {subject_id}')
        return None
    
    print(f'  Fitting {subject_id}: {len(func_imgs)} runs found')
    
    # Define the first-level model
    first_level_model = FirstLevelModel(
        t_r=tr,
        slice_time_ref=0.5,          # middle of TR (fMRIPrep default)
        hrf_model='spm',             # canonical HRF
        drift_model='cosine',        # high-pass filter via cosines
        high_pass=0.01,              # 100s cutoff
        smoothing_fwhm=smoothing_fwhm,  # spatial smoothing (mm)
        noise_model='ar1',           # AR(1) autocorrelation model
        standardize=False,
        signal_scaling=0,            # percent signal change
        minimize_memory=True
    )
    
    # Fit the model
    first_level_model.fit(
        run_imgs=func_imgs,
        events=events_list,
        confounds=confounds_list
    )
    
    return first_level_model


print('First-level fitting function defined.')

In [ ]:
# ========================================
# Fit first-level model for subject 01
# ========================================
# NOTE: This requires the actual data to be downloaded.
# If data is not available, the function will print a warning.

sub01_model = fit_first_level(
    subject_id='sub-01',
    raw_dir=RAW_DIR,
    deriv_dir=DERIV_DIR,
    task=TASK,
    n_runs=N_RUNS,
    smoothing_fwhm=6.0,
    tr=2.0
)

### 5.2 Define Contrasts

Contrasts test specific hypotheses about the β (beta) parameters:

| Contrast | What it tests |
|----------|---------------|
| **gain** | Brain regions that respond to gain amount |
| **loss** | Brain regions that respond to loss amount |
| **loss > gain** | Regions more sensitive to losses than gains (neural loss aversion) |
| **gain > loss** | Regions more sensitive to gains than losses |

In [ ]:
# Define contrasts
# These are expressed as dictionaries mapping condition names to weights

contrasts = {
    'gain_effect':       'gain',                # positive effect of gain
    'loss_effect':       'loss',                # positive effect of loss  
    'loss_minus_gain':   'loss - gain',         # neural loss aversion
    'gain_minus_loss':   'gain - loss',         # gain preference
    'gamble_vs_rest':    'gamble',              # any gamble vs. baseline
}

print('Defined contrasts:')
for name, expr in contrasts.items():
    print(f'  {name:20s} → {expr}')

In [ ]:
# Compute contrast maps for subject 01
if sub01_model is not None:
    print('Computing contrasts for sub-01...\n')
    
    fig, axes = plt.subplots(len(contrasts), 1, figsize=(12, 4*len(contrasts)))
    
    for i, (name, expression) in enumerate(contrasts.items()):
        # Compute the z-map for this contrast
        z_map = sub01_model.compute_contrast(expression, output_type='z_score')
        
        # Plot the thresholded z-map
        plotting.plot_glass_brain(
            z_map,
            threshold=2.3,          # z > 2.3 (p < 0.01 uncorrected)
            title=f'{name} (z > 2.3)',
            axes=axes[i],
            display_mode='lyrz',
            colorbar=True,
            plot_abs=False
        )
    
    plt.tight_layout()
    plt.show()
else:
    print('Skipping — model not fitted (data not downloaded).')
    print('Once data is available, this cell will show z-score maps for each contrast.')

In [ ]:
# Detailed view of the key contrast: neural loss aversion
if sub01_model is not None:
    z_loss_aversion = sub01_model.compute_contrast('loss - gain', output_type='z_score')
    
    # Interactive stat map
    view = plotting.view_img(
        z_loss_aversion, 
        threshold=2.3,
        title='Sub-01: Loss > Gain (Neural Loss Aversion)'
    )
    view  # displays interactive viewer in Jupyter
else:
    print('Skipping — model not fitted.')

In [ ]:
# Generate an HTML report for the first-level model
if sub01_model is not None:
    report = make_glm_report(
        sub01_model,
        contrasts=contrasts,
        title='First-Level GLM Report: sub-01',
        height_control='fpr',    # false positive rate
        alpha=0.001
    )
    # Save or display the report
    report.save_as_html('sub01_first_level_report.html')
    print('Report saved to: sub01_first_level_report.html')
    # report  # uncomment to display inline

---
## 6. First-Level for All Subjects

Now we loop over all subjects to generate contrast maps for the group analysis.

In [ ]:
# Output directory for first-level contrast maps
FIRST_LEVEL_DIR = os.path.join(DATA_DIR, 'first_level_output')
os.makedirs(FIRST_LEVEL_DIR, exist_ok=True)

# Store contrast maps for second-level analysis
second_level_input = {name: [] for name in contrasts}
successful_subjects = []

for subject_id in SUBJECTS:
    print(f'\n--- Processing {subject_id} ---')
    
    model = fit_first_level(
        subject_id=subject_id,
        raw_dir=RAW_DIR,
        deriv_dir=DERIV_DIR,
        task=TASK,
        n_runs=N_RUNS,
        smoothing_fwhm=6.0,
        tr=2.0
    )
    
    if model is None:
        print(f'  Skipping {subject_id}')
        continue
    
    successful_subjects.append(subject_id)
    
    # Compute and save contrast maps
    sub_dir = os.path.join(FIRST_LEVEL_DIR, subject_id)
    os.makedirs(sub_dir, exist_ok=True)
    
    for name, expression in contrasts.items():
        # Compute contrast (effect size map, not z-score, for 2nd level)
        effect_map = model.compute_contrast(expression, output_type='effect_size')
        
        # Save the map
        out_file = os.path.join(sub_dir, f'{name}_effect.nii.gz')
        effect_map.to_filename(out_file)
        
        # Store for second-level
        second_level_input[name].append(out_file)
    
    print(f'  Saved {len(contrasts)} contrast maps')

print(f'\n=== First-level complete ===')
print(f'Successful subjects: {len(successful_subjects)}/{len(SUBJECTS)}')

---
## 7. Second-Level GLM (Group Analysis)

### What is second-level analysis?

First-level gives us a contrast map **per subject**. Second-level tests whether the effect is **consistent across the group**.

The simplest group analysis is a **one-sample t-test**:
- For each voxel, test H₀: mean β = 0 across subjects
- This tells us: "Is this brain region reliably activated across the group?"

$$t = \frac{\bar{\beta}}{SE(\bar{\beta})}$$

In [ ]:
from nilearn.glm.second_level import SecondLevelModel
from nilearn.glm import threshold_stats_img

def run_second_level(contrast_imgs, contrast_name, height_control='fpr', alpha=0.001):
    """
    Run a one-sample t-test at the group level.
    
    Parameters:
        contrast_imgs: list of first-level effect size maps (one per subject)
        contrast_name: name of the contrast (for labeling)
        height_control: multiple comparison correction method
                       'fpr' = false positive rate (uncorrected)
                       'fdr' = false discovery rate
                       'bonferroni' = Bonferroni correction
        alpha: significance threshold
    
    Returns:
        z_map: thresholded z-score map
        threshold: computed threshold value
    """
    print(f'  Second-level: {contrast_name} (n={len(contrast_imgs)} subjects)')
    
    # Design matrix for one-sample t-test: just a column of ones
    design_matrix = pd.DataFrame({'intercept': np.ones(len(contrast_imgs))})
    
    # Fit the second-level model
    second_level_model = SecondLevelModel(smoothing_fwhm=None)  # already smoothed
    second_level_model.fit(
        second_level_input=contrast_imgs,
        design_matrix=design_matrix
    )
    
    # Compute the z-map
    z_map = second_level_model.compute_contrast(
        second_level_contrast='intercept',
        output_type='z_score'
    )
    
    # Apply threshold with multiple comparison correction
    thresholded_map, threshold = threshold_stats_img(
        z_map,
        alpha=alpha,
        height_control=height_control,
        cluster_threshold=10  # minimum cluster size in voxels
    )
    
    print(f'  Threshold (z): {threshold:.2f} ({height_control}, α={alpha})')
    
    return z_map, thresholded_map, threshold, second_level_model


print('Second-level function defined.')

In [ ]:
# ============================================
# Run second-level analysis for all contrasts
# ============================================

second_level_results = {}

if successful_subjects:
    print('Running group-level analyses...\n')
    
    for name in contrasts:
        if second_level_input[name]:
            z_map, thresh_map, threshold, model = run_second_level(
                contrast_imgs=second_level_input[name],
                contrast_name=name,
                height_control='fdr',   # FDR correction
                alpha=0.05              # q < 0.05
            )
            second_level_results[name] = {
                'z_map': z_map,
                'thresholded_map': thresh_map,
                'threshold': threshold,
                'model': model
            }
    
    print(f'\nCompleted {len(second_level_results)} group-level contrasts.')
else:
    print('No subjects processed. Please download the data first (see Section 1).')
    print('\nWhen data is available, this will run one-sample t-tests for each contrast.')

### 7.1 Visualize Group Results

In [ ]:
# Plot all group-level contrasts
if second_level_results:
    n_contrasts = len(second_level_results)
    fig, axes = plt.subplots(n_contrasts, 1, figsize=(12, 4 * n_contrasts))
    if n_contrasts == 1:
        axes = [axes]
    
    for i, (name, result) in enumerate(second_level_results.items()):
        plotting.plot_glass_brain(
            result['thresholded_map'],
            threshold=0,  # already thresholded
            title=f'GROUP: {name} (FDR q<0.05, z>{result["threshold"]:.2f})',
            axes=axes[i],
            display_mode='lyrz',
            colorbar=True,
            plot_abs=False
        )
    
    plt.tight_layout()
    plt.show()
else:
    print('No results to display.')

In [ ]:
# Detailed view of neural loss aversion at group level
if 'loss_minus_gain' in second_level_results:
    result = second_level_results['loss_minus_gain']
    
    # Stat map on anatomical background
    fig, axes = plt.subplots(1, 1, figsize=(14, 5))
    plotting.plot_stat_map(
        result['thresholded_map'],
        title='GROUP: Neural Loss Aversion (Loss > Gain, FDR q<0.05)',
        display_mode='z',
        cut_coords=[-10, 0, 10, 20, 30, 40, 50],
        threshold=0,
        colorbar=True,
        axes=axes
    )
    plt.show()
    
    # Interactive viewer
    view = plotting.view_img(
        result['z_map'],
        threshold=result['threshold'],
        title='GROUP: Neural Loss Aversion (Loss > Gain)'
    )
    view

### 7.2 Extract Peak Coordinates and Cluster Table

In [ ]:
from nilearn.reporting import get_clusters_table

if 'loss_minus_gain' in second_level_results:
    result = second_level_results['loss_minus_gain']
    
    # Get cluster table
    table = get_clusters_table(
        result['z_map'],
        stat_threshold=result['threshold'],
        cluster_threshold=10,     # minimum 10 voxels
        two_sided=True            # show both positive and negative
    )
    
    print('=== Cluster Table: Loss > Gain ===')
    print(f'Threshold: z > {result["threshold"]:.2f}\n')
    display(table)
else:
    print('No results available.')
    print('\nExpected regions for Loss > Gain (from Tom et al., 2007):')
    print('  - Ventral striatum (bilateral)')
    print('  - Ventromedial prefrontal cortex (vmPFC)')
    print('  - Anterior insula')
    print('  - Medial prefrontal cortex')

---
## 8. Generate Second-Level Report

In [ ]:
# Generate a comprehensive HTML report for the group analysis
if second_level_results:
    for name, result in second_level_results.items():
        report = make_glm_report(
            result['model'],
            contrasts='intercept',
            title=f'Second-Level Report: {name}',
            height_control='fdr',
            alpha=0.05,
            cluster_threshold=10
        )
        report_file = os.path.join(FIRST_LEVEL_DIR, f'group_{name}_report.html')
        report.save_as_html(report_file)
        print(f'Saved: {report_file}')
    
    print('\nAll reports generated!')
else:
    print('No results to report.')

---
## 9. Summary & Interpretation

### Pipeline Overview

```
Raw BIDS data (OpenNeuro ds000005)
        │
        ▼
  fMRIPrep (already done)
  • Motion correction
  • Spatial normalization → MNI space
  • Confound estimation
        │
        ▼
  First-Level GLM (per subject)
  • Spatial smoothing (6mm FWHM)
  • Design matrix: gamble + gain (parametric) + loss (parametric)
  • Confound regressors: 6 motion + WM + CSF
  • High-pass filter: cosine drift (100s cutoff)
  • Contrasts: gain, loss, loss−gain, gain−loss
        │
        ▼
  Second-Level GLM (group)
  • One-sample t-test across 16 subjects
  • FDR correction (q < 0.05)
  • Cluster table with peak coordinates
```

### Expected Results (from Tom et al., 2007)

| Region | Gain Effect | Loss Effect | Neural Loss Aversion |
|--------|:-----------:|:-----------:|:-------------------:|
| **Ventral striatum** | ↑ with gain | ↓ with loss | Loss slope > Gain slope |
| **vmPFC** | ↑ with gain | ↓ with loss | Loss slope > Gain slope |
| **Anterior insula** | ↑ with gain | ↓ with loss | Loss slope > Gain slope |
| **Dorsal striatum** | ↑ with gain | — | — |

### Key Takeaway

Gains and losses are represented in **overlapping brain circuits** (not separate "pleasure" and "pain" systems).  
The **steeper slope** for losses reflects the behavioral finding that "losses loom larger than gains" (Kahneman & Tversky, 1979) — this is **neural loss aversion**.

---

### References

- Tom SM, Fox CR, Trepel C, Poldrack RA (2007). The neural basis of loss aversion in decision-making under risk. *Science*, 315(5811):515-8.
- Kahneman D, Tversky A (1979). Prospect Theory: An Analysis of Decision under Risk. *Econometrica*, 47(2):263-91.
- Esteban O et al. (2019). fMRIPrep: a robust preprocessing pipeline for functional MRI. *Nature Methods*, 16:111-116.
- Abraham A et al. (2014). Machine learning for neuroimaging with scikit-learn. *Frontiers in Neuroinformatics*, 8:14.